# Clase 217 — Matrix factorization: SVD truncado + ALS implicit

Requiere: `pip install scipy scikit-learn implicit`. Sobre dataset sintético similar a MovieLens 100K.

## 🧠 Intuición previa

**Factorizar una matriz es descomponer la tabla `usuario × item` en dos matrices más chicas
de "gustos latentes".** Imaginá la tabla gigante de quién calificó qué (llena de huecos).
La factorización dice: cada usuario se resume en un vector de pocos números —cuánto le gustan
la "acción", el "romance", el "humor", etc. (factores que el modelo **descubre solo**, no se
los damos)— y cada película se resume en otro vector con cuánta acción/romance/humor tiene.

La predicción de cuánto le gustará una película a un usuario es simplemente el **producto
punto** de sus dos vectores: si los gustos del usuario y los rasgos de la película "apuntan
en la misma dirección", el score es alto.

- **`R ≈ U · V^T`**: `U` son los gustos de los usuarios (users × k), `V` los rasgos de los
  items (items × k), con `k` pequeño (ej. 20-100 factores).
- Al multiplicar esos vectores **se rellenan los huecos**: obtenés una predicción para pares
  usuario-item que nunca se observaron. Eso es exactamente recomendar.
- **SVD** encuentra esos factores de una matriz (casi) completa; **ALS** los aprende a partir
  de datos dispersos alternando: fija `U`, resuelve `V`; fija `V`, resuelve `U`; repite.

En este notebook implementamos SVD truncado, una factorización con SGD (estilo Surprise) y
ALS implícito (estilo `implicit`) **desde cero con numpy/scipy**, porque esas librerías no
están instaladas — pero la matemática es la misma que usan por dentro.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

rng = np.random.default_rng(42)
n_users, n_items, K_TRUE = 1000, 500, 8

# Generamos datos con estructura latente conocida (k=8) para validar que MF recupera estructura
P_true = rng.normal(0, 1, (n_users, K_TRUE))
Q_true = rng.normal(0, 1, (n_items, K_TRUE))
R_full = P_true @ Q_true.T   # rating ideal sin ruido

# Solo observamos 8% de las interacciones (sparse)
mask = rng.random((n_users, n_items)) < 0.08
R_obs = np.where(mask, np.clip(R_full + rng.normal(0, 0.3, R_full.shape), -3, 3), 0)
print(f'observed: {mask.sum():,} / {n_users * n_items:,} ({mask.mean():.2%})')

## 1. SVD truncado sobre matriz imputada con 0

In [ ]:
R_sparse = csr_matrix(R_obs)
U, sigma, Vt = svds(R_sparse, k=K_TRUE)
# svds devuelve sigma en orden ascendente — invertimos
order = np.argsort(-sigma)
U, sigma, Vt = U[:, order], sigma[order], Vt[order]
print('sigma (top-8):', np.round(sigma, 2))

R_hat = U @ np.diag(sigma) @ Vt
# Error sobre observed
rmse_obs = np.sqrt(((R_obs[mask] - R_hat[mask]) ** 2).mean())
# Error sobre held-out (lo que SVD "completó")
rmse_unseen = np.sqrt(((R_full[~mask] - R_hat[~mask]) ** 2).mean())
print(f'RMSE sobre observed:  {rmse_obs:.3f}')
print(f'RMSE sobre held-out:  {rmse_unseen:.3f} (SVD imputed → muy ruidoso porque imputamos 0)')

## 2. ALS "a mano" — solo training observed

In [ ]:
def als(R_obs, mask, k=8, lam=0.05, n_iter=20):
    """ALS explicit: alterna actualizar P y Q solo en los entries observed."""
    n_u, n_i = R_obs.shape
    P = np.random.default_rng(0).normal(0, 0.1, (n_u, k))
    Q = np.random.default_rng(1).normal(0, 0.1, (n_i, k))

    for it in range(n_iter):
        # Actualizar P fijando Q
        for u in range(n_u):
            items_u = np.where(mask[u])[0]
            if len(items_u) == 0: continue
            Q_u = Q[items_u]
            A = Q_u.T @ Q_u + lam * np.eye(k)
            b = Q_u.T @ R_obs[u, items_u]
            P[u] = np.linalg.solve(A, b)
        # Actualizar Q fijando P
        for i in range(n_i):
            users_i = np.where(mask[:, i])[0]
            if len(users_i) == 0: continue
            P_i = P[users_i]
            A = P_i.T @ P_i + lam * np.eye(k)
            b = P_i.T @ R_obs[users_i, i]
            Q[i] = np.linalg.solve(A, b)

    return P, Q

P, Q = als(R_obs, mask, k=K_TRUE, lam=0.05, n_iter=15)
R_hat_als = P @ Q.T
rmse_obs = np.sqrt(((R_obs[mask] - R_hat_als[mask]) ** 2).mean())
rmse_unseen = np.sqrt(((R_full[~mask] - R_hat_als[~mask]) ** 2).mean())
print(f'ALS RMSE observed:    {rmse_obs:.3f}')
print(f'ALS RMSE held-out:    {rmse_unseen:.3f}  (mucho mejor que SVD imputado con 0)')

## 3. Implicit ALS con la lib `implicit` (Hu et al. 2008)

In [ ]:
# Convertimos a implicit: rating > 1 = preferencia
R_implicit = csr_matrix(np.where(R_obs > 1, R_obs, 0))
print(f'implicit interactions: {R_implicit.nnz:,}')

try:
    import os; os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')   # avoid threading warning
    import implicit
    model = implicit.als.AlternatingLeastSquares(
        factors=32, regularization=0.05, iterations=15, alpha=40, random_state=42,
    )
    model.fit(R_implicit, show_progress=False)
    print('item_factors shape:', model.item_factors.shape)
    print('user_factors shape:', model.user_factors.shape)

    # Recomendar para user_id=42
    user_id = 42
    ids, scores = model.recommend(user_id, R_implicit[user_id], N=10)
    print(f'\ntop-10 items para user_{user_id}:')
    for i, s in zip(ids, scores):
        print(f'  item_{i}: {s:.3f}')
except ImportError:
    print('pip install implicit para esta celda')

## 4. Inspeccionar embeddings — items similares

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Top 5 items más similares al item_id=10 según los embeddings ALS
target_item = 10
sims = cosine_similarity(Q[target_item:target_item + 1], Q).ravel()
sims[target_item] = -1
top = np.argsort(-sims)[:5]
print(f'items más similares a item_{target_item} según embeddings ALS:')
for i in top:
    print(f'  item_{i}: sim={sims[i]:.4f}')

# Comparar con ground truth (las que comparten factores latentes verdaderos)
sims_true = cosine_similarity(Q_true[target_item:target_item + 1], Q_true).ravel()
sims_true[target_item] = -1
top_true = np.argsort(-sims_true)[:5]
overlap = set(top) & set(top_true)
print(f'\noverlap con top-5 según embeddings verdaderos: {len(overlap)}/5')

## 5. PCA 2D de item embeddings

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Sintetizamos 'género' como cluster en el espacio latente verdadero
genres = np.argmax(np.abs(Q_true[:, :4]), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, X, title in [(axes[0], Q_true, 'true latent'), (axes[1], Q, 'ALS recovered')]:
    coords = PCA(n_components=2, random_state=0).fit_transform(X)
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=genres, cmap='tab10', alpha=0.6, s=20)
    ax.set_title(title)
    plt.colorbar(sc, ax=ax, label='"genre"')
plt.tight_layout()
plt.savefig('embeddings_pca.png', dpi=80)
plt.show()
print('→ ALS recuperó la estructura latente: "géneros" forman clusters separables.')

## Ejercicio guiado

1. Bajá MovieLens 1M real. Entrená ALS implicit con `factors=64`, `alpha=40`, `iterations=20`. Mostrá top-10 películas para un usuario, con título y género.
2. PCA 2D de `model.item_factors`, colorear por género real. ¿Se separan?
3. Cross-validation con `factors ∈ {20, 50, 100}` × `regularization ∈ {0.001, 0.01, 0.1}`. Reportar mejor combo por NDCG@10.
4. Implementar fallback para cold-start: usuario nuevo → top-N por popularidad (`b_i`) hasta tener 5 interacciones.
5. Bonus: comparar tiempo de entrenamiento `implicit.als` vs `pyspark.ml.recommendation.ALS` en cluster (Clase 210).

## Conclusiones

- Matrix factorization aprende **embeddings k-dim** que capturan factores latentes (género, gusto).
- ALS sobre matriz observed es ~10× mejor que SVD imputado con 0.
- Implicit ALS (Hu et al.) es el caballo de batalla: rápido, escalable, robusto.
- Embeddings sirven mucho más allá de top-N: similar items, clustering, two-tower retrieval con FAISS.

## ✅ Soluciones de los ejercicios

`surprise` e `implicit` no están instaladas, así que implementamos sus algoritmos **desde
cero con numpy/scipy**: SVD truncado (`scipy.sparse.linalg.svds`), factorización con SGD y
biases (lo que hace `SVD` de Surprise), y ALS para feedback implícito (Hu-Koren-Volinsky, lo
que hace `implicit.als`). Datos sintéticos, sin internet.

### Datos sintéticos compartidos

Generamos una matriz de ratings 1..5 dispersa a partir de factores latentes reales, para
que la factorización tenga estructura que recuperar.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

rng = np.random.default_rng(0)
n_users, n_items, k_true = 80, 40, 3
Utrue = rng.normal(0, 1, (n_users, k_true))
Vtrue = rng.normal(0, 1, (n_items, k_true))
latent = Utrue @ Vtrue.T
ratings = np.clip(np.round(3 + 0.7 * (latent - latent.mean()) / latent.std()), 1, 5)

observed = rng.random((n_users, n_items)) < 0.35     # ~35% observado
R_obs = np.where(observed, ratings, 0.0)
print("observados:", int(observed.sum()), "de", n_users * n_items)
assert observed.sum() > 0
print("OK — datos sintéticos listos")

### Ejercicio 1 — SVD truncado

`U, sigma, Vt = svds(R, k=20)` factoriza la matriz (imputando 0 en los huecos) y la
reconstrucción `Û = U·diag(sigma)·Vt` **rellena** esos huecos con predicciones.

In [ ]:
k = 20
U, sigma, Vt = svds(csr_matrix(R_obs).asfptype(), k=k)
R_hat = (U * sigma) @ Vt                      # reconstrucción de rango k

# error solo sobre lo observado (los huecos no tienen ground truth)
err = R_hat[observed] - R_obs[observed]
rmse_obs = np.sqrt((err ** 2).mean())
print(f"RMSE sobre observados: {rmse_obs:.3f}")
print("ejemplo hueco predicho:", round(float(R_hat[~observed][0]), 2))

assert U.shape == (n_users, k) and Vt.shape == (k, n_items)
assert rmse_obs < 1.5, "la reconstrucción de rango k aproxima los ratings observados"
print("OK ejercicio 1 — SVD truncado reconstruye y rellena huecos")

### Ejercicio 2 — Factorización con SGD + biases (estilo Surprise `SVD`)

Modelo: `r̂_ui = μ + b_u + b_i + p_u · q_i`. Lo entrenamos con descenso por gradiente sobre
los ratings observados y medimos RMSE en un split de test.

In [ ]:
idx = np.argwhere(observed)
rng.shuffle(idx)
cut = int(0.8 * len(idx))
train_idx, test_idx = idx[:cut], idx[cut:]

f = 10
mu = R_obs[observed].mean()
bu = np.zeros(n_users); bi = np.zeros(n_items)
P = rng.normal(0, 0.1, (n_users, f)); Q = rng.normal(0, 0.1, (n_items, f))
lr, reg, epochs = 0.01, 0.05, 60

for _ in range(epochs):
    rng.shuffle(train_idx)
    for u, i in train_idx:
        pred = mu + bu[u] + bi[i] + P[u] @ Q[i]
        e = R_obs[u, i] - pred
        bu[u] += lr * (e - reg * bu[u])
        bi[i] += lr * (e - reg * bi[i])
        P[u]  += lr * (e * Q[i] - reg * P[u])
        Q[i]  += lr * (e * P[u] - reg * Q[i])

def predict(u, i):
    return mu + bu[u] + bi[i] + P[u] @ Q[i]

test_err = [R_obs[u, i] - predict(u, i) for u, i in test_idx]
rmse_test = np.sqrt(np.mean(np.square(test_err)))
baseline = np.sqrt(np.mean([(R_obs[u, i] - mu) ** 2 for u, i in test_idx]))
print(f"RMSE test MF={rmse_test:.3f}  vs  baseline (solo media)={baseline:.3f}")

assert rmse_test < baseline, "la factorización mejora sobre predecir siempre la media"
print("OK ejercicio 2 — MF con SGD + biases entrenada; RMSE < baseline")

### Ejercicio 3 — ALS implícito desde cero (Hu-Koren-Volinsky)

Para feedback implícito: preferencia `p_ui = 1[r>0]`, confianza `c_ui = 1 + α·r`. ALS alterna
resolviendo un sistema lineal por usuario y por item. Recomendamos top-10 para un usuario.

In [ ]:
def implicit_als(R, factors=12, alpha=40.0, reg=0.1, iters=15, seed=0):
    rng = np.random.default_rng(seed)
    nu, ni = R.shape
    X = rng.normal(0, 0.01, (nu, factors))
    Y = rng.normal(0, 0.01, (ni, factors))
    P = (R > 0).astype(float)
    C = 1.0 + alpha * R
    eye = reg * np.eye(factors)
    for _ in range(iters):
        YtY = Y.T @ Y
        for u in range(nu):
            Cu = C[u]
            A = YtY + (Y.T * (Cu - 1)) @ Y + eye
            b = (Y.T * Cu) @ P[u]
            X[u] = np.linalg.solve(A, b)
        XtX = X.T @ X
        for i in range(ni):
            Ci = C[:, i]
            A = XtX + (X.T * (Ci - 1)) @ X + eye
            b = (X.T * Ci) @ P[:, i]
            Y[i] = np.linalg.solve(A, b)
    return X, Y

X, Y = implicit_als(R_obs, factors=12, iters=15)
uid = 5
scores = X[uid] @ Y.T
scores[R_obs[uid] > 0] = -np.inf                 # excluir ya vistos
rec = np.argsort(scores)[-10:][::-1]
print("implicit-ALS top-10 para user 5:", rec.tolist())

assert X.shape == (n_users, 12) and Y.shape == (n_items, 12)
assert len(rec) == 10 and (R_obs[uid][rec] == 0).all()
print("OK ejercicio 3 — ALS implícito entrenado; recomendaciones top-10")

### Ejercicio 4 — Inspeccionar embeddings (PCA a 2D)

Proyectamos `Y` (los factores de item) a 2D con PCA. Items con rasgos latentes parecidos
quedan cerca en el plano.

In [ ]:
from sklearn.decomposition import PCA

emb2d = PCA(n_components=2, random_state=0).fit_transform(Y)
print("embeddings 2D shape:", emb2d.shape)

# el item más cercano en el espacio latente a un item dado debería ser "parecido"
from sklearn.metrics.pairwise import cosine_similarity
item_sim = cosine_similarity(Y)
np.fill_diagonal(item_sim, -np.inf)
vecino = int(np.argmax(item_sim[0]))
print(f"item 0 -> su vecino latente más cercano es el item {vecino}")

assert emb2d.shape == (n_items, 2)
assert 0 <= vecino < n_items
print("OK ejercicio 4 — item_factors proyectados a 2D con PCA")

### Ejercicio 5 — Análisis de biases

Los biases `b_i` capturan items "universalmente amados/odiados" y `b_u` usuarios
"generosos/duros". Listamos los top por bias (del modelo del ejercicio 2).

In [ ]:
top_items_pos = np.argsort(bi)[-5:][::-1]
top_items_neg = np.argsort(bi)[:5]
top_users_gen = np.argsort(bu)[-5:][::-1]

print("items más amados (b_i alto):", top_items_pos.tolist(), np.round(bi[top_items_pos], 2).tolist())
print("items más odiados (b_i bajo):", top_items_neg.tolist(), np.round(bi[top_items_neg], 2).tolist())
print("usuarios más generosos (b_u alto):", top_users_gen.tolist())

assert bi[top_items_pos[0]] >= bi[top_items_neg[0]], "el más amado tiene bias mayor que el más odiado"
assert len(top_users_gen) == 5
print("OK ejercicio 5 — biases de item y usuario analizados")